# Inspect and repair MonoTM feature labels

Install MonoTM in this notebook's Python environment first (`pip install -e .` from the repository). Set the paths below to an existing run and the CSV used to fit it. Inspection needs no API key. Retry cells make real requests to your configured provider; export `MONOTM_API_KEY` or `DEEPINFRA_API_KEY` before launching Jupyter.

These cells reuse the saved interpretation activations and Θ. They do not train SAEs or LDA. Completed validation rebuilds only the semantic descriptors B and reports.


In [ ]:
from pathlib import Path
from monotm import inspect_run, retry_features

RUN_DIR = Path("../runs/my_corpus")
INPUT_CSV = Path("../my_documents.csv")
FEATURE_ID = None  # Set after reviewing the candidate IDs below.


## 1. Find failed or low-scoring features

`validated` means prediction scoring completed; check the score against the cutoff. `--failed`/`failed=True` targets unfinished requests, not low IS.


In [ ]:
summary = inspect_run(RUN_DIR)
print("Run status:", summary["status"], "IS cutoff:", summary["is_threshold"])
print("Low-scoring IDs:", summary["low_score_feature_ids"])
[(r["feature_id"], r["status"], round(r.get("is_score", 0), 3), r.get("label", ""))
 for r in summary["features"]]


## 2. Read a feature's prompt and raw responses

Set `FEATURE_ID` above and run this cell. Attempt history includes failed responses and identifies Interpreter/Predictor requests and validation documents. Saved text includes corpus excerpts; keep diagnostics private.


In [ ]:
assert FEATURE_ID is not None, "Set FEATURE_ID to an ID from the summary first."
details = inspect_run(RUN_DIR, feature_id=FEATURE_ID)
details["features"][0]


In [ ]:
for attempt in details["attempts"]:
    print("CONTEXT:", attempt["context"])
    print("MODEL:", attempt["request"]["model"], "STATUS:", attempt["status"])
    print("REQUEST PARAMETERS:", {k: v for k, v in attempt["request"].items() if k != "messages"})
    print("TOKEN USAGE:", attempt.get("usage"))
    print("FINISH REASON:", attempt.get("finish_reason"))
    print("PROMPT:\n", attempt["request"]["messages"][0]["content"])
    print("RESPONSE:", attempt.get("response"))
    print("RAW HTTP BODY:", attempt.get("raw_response"))
    print("ERROR:", attempt.get("error"), "\n")


## 3. Retry this feature and validate it again

Leave model names as `None` to use the models recorded in the run, or supply replacement model IDs. Explicit feature selection makes fresh requests by default, so a well-formed but poor cached label can be replaced. Replies with `finish_reason="length"` are recorded as truncated and stop without automatic retries or budget changes. Increase the affected role's `token_limit`, or select a supported lower `reasoning_effort`, to retry. Generation overrides are partial dictionaries merged with the original run settings. Use `{"temperature": None}` to omit temperature; `{"reasoning_effort": None}` omits effort while `{"reasoning_effort": "none"}` explicitly requests none. Choose `token_limit_parameter="max_completion_tokens"` for models requiring it. For another endpoint, also pass `base_url` and set its `MONOTM_API_KEY`.

To retain the label and change only validation, use `stage="predictor"` and omit the Interpreter override. To set a label yourself, pass `label="your description"`; Predictor validation still runs. Each retry archives the preceding results under `cache/repairs/`.


In [ ]:
assert FEATURE_ID is not None, "Set FEATURE_ID first."
INTERPRETER_MODEL = None  # Example: another model supported by your provider.
PREDICTOR_MODEL = None
updated = retry_features(
    RUN_DIR, INPUT_CSV, feature_ids=[FEATURE_ID],
    interpreter_model=INTERPRETER_MODEL, predictor_model=PREDICTOR_MODEL,
    interpreter_generation={"token_limit": 2048},
    predictor_generation={"token_limit": 2048},
)


If this raises because other candidates remain unfinished, the selected feature's progress is still saved. Continue them with `retry_features(RUN_DIR, INPUT_CSV, failed=True)`; repeat model/endpoint overrides when needed. Accepted matching responses are reused with `failed=True`. The same APIs are available as `monotm inspect` and `monotm retry` from the terminal.


In [ ]:
after = inspect_run(RUN_DIR, feature_id=FEATURE_ID)
after["features"][0]
